# TRM Compiler Pass Ordering — Synthetic Environment

Train a 60K-param TRM to order LLVM optimization passes using synthetic environment.

**Note:** If you want real LLVM via CompilerGym, you need Python 3.11 (not available on Colab). 
This notebook uses the realistic synthetic environment that models real LLVM behavior.

**Runtime:** CPU is sufficient, GPU will be faster.

In [ ]:
!pip install -q torch numpy matplotlib

In [ ]:
import os

# CHANGE THIS to your repo URL
REPO_URL = 'https://github.com/YOUR_USERNAME/trm-youtubevids.git'
PROJECT_DIR = '/content/trm-youtubevids'

if not os.path.exists(PROJECT_DIR):
    !git clone $REPO_URL $PROJECT_DIR
else:
    !cd $PROJECT_DIR && git pull

os.chdir(PROJECT_DIR)

## Config

In [ ]:
import torch
from pathlib import Path

CONFIG = {
    'benchmarks': [
        'qsort', 'adpcm', 'blowfish', 'bzip2', 'dijkstra', 'sha',
        'gsm', 'ispell', 'jpeg-c', 'lame', 'patricia', 'rijndael',
        'stringsearch', 'susan', 'tiff2bw', 'tiff2rgba', 'tiffdither', 'tiffmedian'
    ],
    'episodes_per_benchmark': 50,
    'max_steps_per_episode': 30,
    'latent_dim': 64,
    'hidden_dim': 128,
    'n_recursions': 6,
    'epochs': 30,
    'batch_size': 64,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'eval_benchmarks': ['qsort', 'adpcm', 'blowfish', 'bzip2', 'dijkstra', 'sha'],
    'eval_max_steps': 30,
    'seed': 42,
}

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('trm_compiler_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Generate Traces (Synthetic Environment)

Uses SyntheticCompilerEnv which models real LLVM pass ordering behavior:
- 37 LLVM passes with realistic effectiveness
- 16 pass synergy pairs (instcombine→gvn, loop-unroll→vectorize)
- 4 anti-patterns (harmful orderings)
- 18 benchmark profiles

In [ ]:
import time
from trm_compiler.data import generate_compiler_traces, load_traces, CompilerTraceDataset
from trm_compiler.env_wrapper import NUM_PASSES

TRACES_PATH = str(OUTPUT_DIR / 'traces_synthetic.json')

if Path(TRACES_PATH).exists():
    print(f'Loading cached traces from {TRACES_PATH}')
    traces = load_traces(TRACES_PATH)
else:
    print('Generating traces with synthetic environment...')
    t0 = time.time()
    traces = generate_compiler_traces(
        benchmarks=CONFIG['benchmarks'],
        episodes_per_benchmark=CONFIG['episodes_per_benchmark'],
        max_steps_per_episode=CONFIG['max_steps_per_episode'],
        use_heuristic=True,  # Use synthetic env (not real LLVM)
        output_path=TRACES_PATH,
        seed=CONFIG['seed'],
    )
    print(f'Generated {len(traces)} traces in {time.time()-t0:.0f}s')

print(f'Total trace records: {len(traces)}')

In [ ]:
import numpy as np
from collections import Counter

bench_counts = Counter(t.benchmark.benchmark_id for t in traces)
rewards = [t.reward for t in traces]

print(f'Total traces: {len(traces)}')
print(f'Benchmarks: {len(bench_counts)}')
print(f'Reward stats: mean={np.mean(rewards):+.4f} std={np.std(rewards):.4f} min={np.min(rewards):+.4f} max={np.max(rewards):+.4f}')

## Train TRM Model

In [ ]:
from torch.utils.data import DataLoader
from trm_compiler.model import TinyPassOrderingRefiner
from trm_compiler.training import train_one_epoch

dataset = CompilerTraceDataset(traces)
dataloader = DataLoader(
    dataset, batch_size=CONFIG['batch_size'],
    shuffle=True, num_workers=2, drop_last=True,
    pin_memory=(DEVICE == 'cuda'),
)

model = TinyPassOrderingRefiner(
    observation_dim=56,
    latent_dim=CONFIG['latent_dim'],
    hidden_dim=CONFIG['hidden_dim'],
    num_passes=NUM_PASSES,
    n_recursions=CONFIG['n_recursions'],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params:,} parameters')
print(f'Dataset: {len(dataset):,} records, {len(dataloader)} batches/epoch')

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])

history = {k: [] for k in ['epoch', 'total_loss', 'pass_loss', 'value_loss', 'halt_loss', 'feasibility_loss']}
best_loss = float('inf')

print(f'\n{"Epoch":>5s} {"Total":>8s} {"Pass":>8s} {"Value":>8s} {"Halt":>8s}')
print('-' * 42)

for epoch in range(CONFIG['epochs']):
    t0 = time.time()
    losses = train_one_epoch(model, dataloader, optimizer, DEVICE)
    scheduler.step()

    for k in history:
        if k == 'epoch':
            history[k].append(epoch + 1)
        else:
            history[k].append(losses[k])

    print(f'{epoch+1:5d} {losses["total_loss"]:8.4f} {losses["pass_loss"]:8.4f} '
          f'{losses["value_loss"]:8.4f} {losses["halt_loss"]:8.4f}  ({time.time()-t0:.1f}s)')

    if losses['total_loss'] < best_loss:
        best_loss = losses['total_loss']
        torch.save({'model_state_dict': model.state_dict(), 'config': CONFIG},
                   str(OUTPUT_DIR / 'trm_model_best.pt'))

torch.save({'model_state_dict': model.state_dict(), 'config': CONFIG},
           str(OUTPUT_DIR / 'trm_model_final.pt'))
print(f'Best loss: {best_loss:.4f}')

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['epoch'], history['total_loss'], 'b-', lw=2)
ax1.set(xlabel='Epoch', ylabel='Loss (log)', title='Total Loss', yscale='log')
ax1.grid(True, alpha=0.3)

for name in ['pass_loss', 'value_loss', 'halt_loss', 'feasibility_loss']:
    ax2.plot(history['epoch'], history[name], label=name.replace('_', ' ').title(), lw=2)
ax2.set(xlabel='Epoch', ylabel='Loss (log)', title='Component Losses', yscale='log')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluate — Synthetic Environment

Compares TRM vs LLVM -Oz, -O3, Random(100), Greedy on synthetic environment.

In [ ]:
from trm_compiler.baselines import run_full_benchmark

model.eval()
run_full_benchmark(
    model,
    benchmarks=CONFIG['eval_benchmarks'],
    max_steps=CONFIG['eval_max_steps'],
    device=DEVICE,
    seed=CONFIG['seed'],
)

## Detailed Rollout — Watch TRM Select Passes

In [ ]:
from trm_compiler.env_wrapper import SyntheticCompilerEnv, pass_id_to_name
from trm_compiler.model import encode_schedule

DETAIL_BENCH = 'qsort'
env = SyntheticCompilerEnv(benchmark_id=DETAIL_BENCH, seed=CONFIG['seed'])
obs, initial_inst = env.reset()

model.eval()
y, z = model.init_latent(1, DEVICE)
feedback_tensor = torch.zeros(1, 4, device=DEVICE)
applied_passes = []
total_reward = 0.0

print(f'{"Step":>4s} {"Pass":<28s} {"Prev":>6s} → {"New":>6s}  {"Reward":>8s}  {"Total":>8s}')
print('-' * 75)

for step in range(CONFIG['eval_max_steps']):
    schedule_vec = encode_schedule(step, CONFIG['eval_max_steps'], applied_passes)
    obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    sch_t = torch.tensor(schedule_vec, dtype=torch.float32, device=DEVICE).unsqueeze(0)

    with torch.no_grad():
        out = model(obs_t, sch_t, feedback_tensor, y, z)
    y, z = out['y'], out['z']

    halt_prob = torch.sigmoid(out['halt_logit']).item()
    if halt_prob > 0.5 and step >= 5:
        print(f'  HALT (prob={halt_prob:.3f})')
        break

    logits = out['pass_logits'].squeeze(0) / 0.8
    for pp in applied_passes[-3:]:
        logits[pp] -= 2.0
    pass_id = torch.multinomial(torch.softmax(logits, dim=-1).clamp(min=1e-6), 1).item()

    prev_inst = env.current_inst_count
    obs, feedback, done, info = env.step(pass_id)
    total_reward += feedback.reward
    applied_passes.append(pass_id)
    feedback_tensor = torch.tensor(feedback.encode(), dtype=torch.float32, device=DEVICE).unsqueeze(0)

    print(f'{step:4d} {pass_id_to_name(pass_id):<28s} {prev_inst:6d} → {env.current_inst_count:6d}  '
          f'{feedback.reward:+8.4f}  {total_reward:+8.4f}')

    if done:
        break

print(f'\n{initial_inst} → {env.current_inst_count} inst  ({(1 - env.current_inst_count/initial_inst)*100:.1f}% reduction)')

In [ ]:
import json

print('Output files:')
for p in sorted(OUTPUT_DIR.iterdir()):
    sz = p.stat().st_size
    print(f'  {p.name:35s} {sz/1e6:.1f} MB' if sz > 1e6 else f'  {p.name:35s} {sz/1e3:.0f} KB')